<a href="https://colab.research.google.com/github/aryanpatel99/GEN-AI-PRACTICE/blob/main/NLP_CLASS_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Noise Reduction and Normalisation

In [ ]:
import string

text = "Ugh ... The deliveries were DELAYED! I seriously hate waiting ... #annoyed"

# Lowercase
text_lower = text.lower()

# remove punctuation using string translation
# Thsi creates a table that swaps punctuation for "Nothing"

translator = str.maketrans("","",string.punctuation)
clean_text  = text_lower.translate(translator)

print("Original:",text)
print("Cleaned:",clean_text)

Original: Ugh ... The deliveries were DELAYED! I seriously hate waiting ... #annoyed
Cleaned: ugh  the deliveries were delayed i seriously hate waiting  annoyed


###str.maketrans has two properties:
1. replace
2. remove

In [ ]:
translator2 = str.maketrans("aeiou","*****")
text = "hello world"

print(text.translate(translator2))

h*ll* w*rld


#**2. Tokenisation**

###**METHOD A - The Simple Way(Split by Space)**

In [ ]:
# the simple way to split is by space
tokens = clean_text.split()

print("Tokens:", tokens)  #gives output - okens: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
print("Token Count:",len(tokens))

Tokens: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Token Count: 10


###**METHOD B - BERT Subword Tokenization**

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

# We will use BERT TOKENIZATION
#  This requires internet to download the vocab

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# complex_text = "The microtransactional system was counterintuitive"

print("Tokens:",tokenizer.tokenize(clean_text))

# output:
# Tokens: ['u', '##gh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokens: ['u', '##gh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']


#**3. Stop Word Removal**

1. For some cases it can flip the meaning, for eg: text = "The movie was NOT good"

  this might remove the not and give the output - movie good(meaning flipped)

(NOT) belongs to stopwords so it will be removed.

->  Dont use it blindly

##----NOTES: IMPORTANT--------
- For **simpler models**(like logistic regression): We **MUST** remove the stop words. These models run on low end devices where memory is limited. We sacrifice a little context to make the model fast and small.

- For **sophisticated models** : We keep the stop words as these models are smart enough to understand that "not" flips the meaning of the sentence.

In [ ]:
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

filtered_tokens = [word for word in tokens if word not in stop_words]

print("Input:", tokens)
print("Filtered Tokens:",filtered_tokens)

Input: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Filtered Tokens: ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


#**4. STEMMING VS LEMMATIZATION**

####We have words like "Deliver" and "waiting". If we train the model on "waiting" and the user tyeps "wait" , the model won't know they are related. We need to cut down tot their root form.

###**Stemming**: A curde chopper (Fast, but sometimes make non-word) (Highly aggressive)
###**Lemmatization**: A dictionary look-up (Accurate) (gives meaningfull words)

In [ ]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["deliveries", "waiting", "delayed", "studied"]

for w in words :
  print(w, "|", stemmer.stem(w), "|", lemmatizer.lemmatize(w))

[nltk_data] Downloading package wordnet to /root/nltk_data...


deliveries | deliveri | delivery
waiting | wait | waiting
delayed | delay | delayed
studied | studi | studied


#**5. VECTORIZATION (THE "BAG OF WORDS")**

###- We still have text strings
###- Math eqns and RNNs cannot multiply strings. So we need to convert them to numbers.

----------------------------------------------------------------------

#**NLP LAB** - 10Feb


In [ ]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Create DataFrame
df = pd.DataFrame({
  "raw_text": [
    "The food was amazing, and the tea was hot!!!",
    "Bad service... I waited 20 minutes for a sandwich?",
    "The samosas are tasty but the coffee is too sweet.",
    "Cleaning was not good, very dirty tables."
    ]
})
df.head()

,raw_text
0,"The food was amazing, and the tea was hot!!!"
1,Bad service... I waited 20 minutes for a sandw...
2,The samosas are tasty but the coffee is too sw...
3,"Cleaning was not good, very dirty tables."


In [ ]:
# 1. Lowercase
df["processed"] = df["raw_text"].str.lower()
df

,raw_text,processed
0,"The food was amazing, and the tea was hot!!!","the food was amazing, and the tea was hot!!!"
1,Bad service... I waited 20 minutes for a sandw...,bad service... i waited 20 minutes for a sandw...
2,The samosas are tasty but the coffee is too sw...,the samosas are tasty but the coffee is too sw...
3,"Cleaning was not good, very dirty tables.","cleaning was not good, very dirty tables."


In [ ]:
# 2. Remove punctuation
# check for apply and lambda
# the first and second params are for replace and the last param is for delete
df["processed"] = df["processed"].apply(lambda x: x.translate(str.maketrans("","",string.punctuation)))
df["processed"]

,processed
0,the food was amazing and the tea was hot
1,bad service i waited 20 minutes for a sandwich
2,the samosas are tasty but the coffee is too sweet
3,cleaning was not good very dirty tables


In [ ]:
df

,raw_text,processed
0,"The food was amazing, and the tea was hot!!!",the food was amazing and the tea was hot
1,Bad service... I waited 20 minutes for a sandw...,bad service i waited 20 minutes for a sandwich
2,The samosas are tasty but the coffee is too sw...,the samosas are tasty but the coffee is too sweet
3,"Cleaning was not good, very dirty tables.",cleaning was not good very dirty tables


In [ ]:
# 3. Tokenisation
df["tokens"] = df['processed'].apply(word_tokenize)
df

,raw_text,processed,tokens
0,"The food was amazing, and the tea was hot!!!",the food was amazing and the tea was hot,"[the, food, was, amazing, and, the, tea, was, ..."
1,Bad service... I waited 20 minutes for a sandw...,bad service i waited 20 minutes for a sandwich,"[bad, service, i, waited, 20, minutes, for, a,..."
2,The samosas are tasty but the coffee is too sw...,the samosas are tasty but the coffee is too sweet,"[the, samosas, are, tasty, but, the, coffee, i..."
3,"Cleaning was not good, very dirty tables.",cleaning was not good very dirty tables,"[cleaning, was, not, good, very, dirty, tables]"


In [ ]:
# 4.stop word removal
stop_words = set(stopwords.words('english'))

# filter tokens
df['no_stops'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])
print(df["no_stops"])
df

0                        [food, amazing, tea, hot]
1    [bad, service, waited, 20, minutes, sandwich]
2                  [samosas, tasty, coffee, sweet]
3                  [cleaning, good, dirty, tables]
Name: no_stops, dtype: object


,raw_text,processed,tokens,no_stops
0,"The food was amazing, and the tea was hot!!!",the food was amazing and the tea was hot,"[the, food, was, amazing, and, the, tea, was, ...","[food, amazing, tea, hot]"
1,Bad service... I waited 20 minutes for a sandw...,bad service i waited 20 minutes for a sandwich,"[bad, service, i, waited, 20, minutes, for, a,...","[bad, service, waited, 20, minutes, sandwich]"
2,The samosas are tasty but the coffee is too sw...,the samosas are tasty but the coffee is too sweet,"[the, samosas, are, tasty, but, the, coffee, i...","[samosas, tasty, coffee, sweet]"
3,"Cleaning was not good, very dirty tables.",cleaning was not good very dirty tables,"[cleaning, was, not, good, very, dirty, tables]","[cleaning, good, dirty, tables]"


In [ ]:
# 5.Lemmatization
lemmatizer = WordNetLemmatizer()

# lemmatize the list of tokens
df["lemmatized"] = df["no_stops"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

# join back into a string for vectorization
df["final_cleaned"] = df['lemmatized'].apply(lambda x: " ".join(x))

df

,raw_text,processed,tokens,no_stops,lemmatized,final_cleaned
0,"The food was amazing, and the tea was hot!!!",the food was amazing and the tea was hot,"[the, food, was, amazing, and, the, tea, was, ...","[food, amazing, tea, hot]","[food, amazing, tea, hot]",food amazing tea hot
1,Bad service... I waited 20 minutes for a sandw...,bad service i waited 20 minutes for a sandwich,"[bad, service, i, waited, 20, minutes, for, a,...","[bad, service, waited, 20, minutes, sandwich]","[bad, service, waited, 20, minute, sandwich]",bad service waited 20 minute sandwich
2,The samosas are tasty but the coffee is too sw...,the samosas are tasty but the coffee is too sweet,"[the, samosas, are, tasty, but, the, coffee, i...","[samosas, tasty, coffee, sweet]","[samosa, tasty, coffee, sweet]",samosa tasty coffee sweet
3,"Cleaning was not good, very dirty tables.",cleaning was not good very dirty tables,"[cleaning, was, not, good, very, dirty, tables]","[cleaning, good, dirty, tables]","[cleaning, good, dirty, table]",cleaning good dirty table


In [ ]:
# 6. Vectorization (bag of words)
cv = CountVectorizer()
bow_matrix = cv.fit_transform(df["final_cleaned"])
# create the final representation dataframe
df_bow = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out())

print("Final Bag of words matrix:")
df_bow


# Notice: they are alphabetically arranged
# semantic order not preserved
# in chatgpt we use transformers


Final Bag of words matrix:


,20,amazing,bad,cleaning,coffee,dirty,food,good,hot,minute,samosa,sandwich,service,sweet,table,tasty,tea,waited
0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0
1,1,0,1,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1
2,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0
3,0,0,0,1,0,1,0,1,0,0,0,0,0,0,1,0,0,0


In [ ]:
df = pd.DataFrame({
  "raw_text": [
    "The food was amazing, and the tea was hot!!!",
    "Bad service... I waited 20 minutes for a sandwich?",
    "The samosas are tasty but the coffee is too sweet.",
    "Cleaning was not good, very dirty tables."
    ]
})



from tensorflow.kers